# 03a · Descărcare Small Landscape Features 2021 prin API-ul CLMS (rulează LOCAL, pe PC)
Familia HRL Small Landscape Features, 2021: **Small Woody Features** (SWF, raster 5 m) și **Woody Vegetation Layer** (WVL, raster 5 m), decupate pe `AOI_BBOX` → `data/raw/slf/` (în afara git). Servesc la expunerea la vânt (03c).

- Proiecția de ieșire: `EPSG:3035` (cea nativă, fără reeșantionare pe server). Trecerea în EPSG:3844 se face la utilizare.
- Autentificare (documentația oficială, eea.github.io/clms-api-docs): cheia de serviciu din `CLMS_KEY_PATH` → JWT semnat RS256 → token de acces (1 h). Nu se afișează nimic secret.
- Reluabil: TaskID-urile se salvează în `data/raw/slf/tasks.json`. O rulare nouă doar verifică starea și descarcă ce e gata.

In [1]:
# Doar local: rădăcina proiectului, .env, constantele cererii
import os, sys, json, time, zipfile
assert 'google.colab' not in sys.modules, 'Rulează pe kernelul local (PC), nu pe Colab.'
if not os.path.exists('gisday_config.py') and os.path.exists('../gisday_config.py'):
    os.chdir('..')
sys.path.insert(0, os.getcwd())
from gisday_config import *
from dotenv import load_dotenv
load_dotenv(ROOT / '.env')
KEY_PATH = ROOT / os.environ.get('CLMS_KEY_PATH', 'data/private/clms_service_key.json')
assert KEY_PATH.exists(), f'Lipsește cheia CLMS: {KEY_PATH.relative_to(ROOT)} (vezi .env.example)'
SLF_DIR = RAW / 'slf'
SLF_DIR.mkdir(parents=True, exist_ok=True)
TASKS = SLF_DIR / 'tasks.json'
API = 'https://land.copernicus.eu/api'
# UID-urile și DatasetDownloadInformationID-urile (colecția raster 5 m), verificate cu GET /api/@search?portal_type=DataSet
DATASETS = {
    'SWF_2021_005m': dict(DatasetID='685a975274554d4ca70f6374fdcc632a',
                          DatasetDownloadInformationID='5a41d86b-c0e0-4d0e-81ab-a8758916e2be'),
    'WVL_2021_005m': dict(DatasetID='e101856cf0af4b799a3cbaa83d4c4fbd',
                          DatasetDownloadInformationID='5a41d86b-c0e0-4d0e-81ab-a8758916e2be'),
}
W, S, E, N = AOI_BBOX
BBOX_CLMS = [W, N, E, S]    # backend-ul citește perechi (lon, lat): colțul NV, apoi colțul SE
OUT_GCS = 'EPSG:3035'
MAX_WAIT_MIN = 45           # cât așteaptă celula de descărcare; peste limită, rulezi din nou mai târziu
print('Lucrez în', os.getcwd(), '| BoundingBox CLMS:', BBOX_CLMS)

Lucrez în C:\Users\lefpa\Downloads\gisday2026\gisday2026 | BoundingBox CLMS: [23.85, 44.0, 24.55, 43.7]


In [2]:
# Token de acces din cheia de serviciu (JWT RS256 -> @@oauth2-token); nimic secret nu se afișează
import jwt, requests
_key = json.loads(KEY_PATH.read_text(encoding='utf-8'))
_session = requests.Session()
_session.headers['Accept'] = 'application/json'

def get_token():
    now = int(time.time())
    claims = {'iss': _key['client_id'], 'sub': _key['user_id'], 'aud': _key['token_uri'],
              'iat': now, 'exp': now + 3600}
    grant = jwt.encode(claims, _key['private_key'], algorithm='RS256')
    r = requests.post(_key['token_uri'], timeout=60,                      # fără antet Authorization aici
                      headers={'Accept': 'application/json', 'Content-Type': 'application/x-www-form-urlencoded'},
                      data={'grant_type': 'urn:ietf:params:oauth:grant-type:jwt-bearer', 'assertion': grant})
    if r.status_code != 200:
        raise RuntimeError(f'Tokenul CLMS a fost refuzat (HTTP {r.status_code}). Verifică cheia; nu încerca ocoliri.')
    _session.headers['Authorization'] = 'Bearer ' + r.json()['access_token']

def api(method, path, **kw):
    """Cerere autentificată; la 401 cere un token nou o singură dată (cum recomandă documentația)."""
    r = _session.request(method, f'{API}/{path}', timeout=120, **kw)
    if r.status_code == 401:
        get_token()
        r = _session.request(method, f'{API}/{path}', timeout=120, **kw)
    return r

get_token()
print('token CLMS obținut (valabil 1 h)')
for name, d in DATASETS.items():
    r = api('GET', '@projections', params={'uid': d['DatasetID']})
    print(name, 'proiecții:', r.status_code, r.text[:300])

token CLMS obținut (valabil 1 h)


SWF_2021_005m proiecții: 200 [
  "EPSG:4326", 
  "EPSG:3035", 
  "EPSG:3857", 
  "EPSG:4258", 
  "EPSG:32740", 
  "EPSG:32622", 
  "EPSG:32620", 
  "EPSG:32738"
]
WVL_2021_005m proiecții: 200 [
  "EPSG:4326", 
  "EPSG:3035", 
  "EPSG:3857", 
  "EPSG:4258", 
  "EPSG:32740", 
  "EPSG:32622", 
  "EPSG:32620", 
  "EPSG:32738"
]


In [3]:
# Trimite câte o cerere per dataset (reluabil: nu retrimite ce are deja TaskID sau fișiere pe disc)
tasks = json.loads(TASKS.read_text(encoding='utf-8')) if TASKS.exists() else {}
for name, d in DATASETS.items():
    if name in tasks or (SLF_DIR / name).exists():
        continue
    body = {'Datasets': [{**d, 'BoundingBox': BBOX_CLMS, 'OutputFormat': 'Geotiff', 'OutputGCS': OUT_GCS}]}
    r = api('POST', '@datarequest_post', json=body)
    print(name, 'POST', r.status_code, r.text[:400])
    r.raise_for_status()
    tasks[name] = r.json()['TaskIds'][0]['TaskID']
    TASKS.write_text(json.dumps(tasks, indent=1), encoding='utf-8')
print('task-uri:', tasks)

task-uri: {'SWF_2021_005m': '21780887710', 'WVL_2021_005m': '97973866919'}


In [4]:
# Așteaptă task-urile și descarcă arhivele gata (DownloadURL e public, valabil 72 h; nu-i trimitem tokenul)
import subprocess

def status(task_id):
    r = api('GET', '@datarequest_status_get', params={'TaskID': task_id})
    r.raise_for_status()
    return r.json()

def fetch(url, dest):
    """Descărcare cu curl (cu User-Agent-ul lui real): serverul FME răspunde 403 clientului python-requests,
    dar servește curl (verificat 06.11.2026). Reîncercările acoperă și intervalul până la publicarea arhivei."""
    subprocess.run(['curl', '-fsSL', '--retry', '6', '--retry-delay', '30', '--retry-all-errors',
                    '-o', str(dest), url], check=True)

pending = {n: t for n, t in tasks.items() if not (SLF_DIR / n).exists()}
t0 = time.time()
while pending:
    for name, tid in list(pending.items()):
        s = status(tid)
        st = s.get('Status')
        print(time.strftime('%H:%M'), name, tid, st)
        if st == 'Finished_ok':
            z = SLF_DIR / f'{name}.zip'
            fetch(s['DownloadURL'], z)
            assert z.stat().st_size == s.get('FileSize', z.stat().st_size), f'{z.name}: dimensiune diferită de FileSize'
            with zipfile.ZipFile(z) as zf:
                zf.extractall(SLF_DIR / name)
            for inner in (SLF_DIR / name).rglob('*.zip'):          # arhive imbricate, dacă există
                with zipfile.ZipFile(inner) as zf:
                    zf.extractall(inner.parent / inner.stem)
            pending.pop(name)
        elif st in ('Finished_nok', 'Rejected', 'Cancelled'):
            raise RuntimeError(f'{name}: task {tid} -> {st}: {s.get("Message", "")}')
    if pending and time.time() - t0 > MAX_WAIT_MIN * 60:
        print(f'Încă în lucru după {MAX_WAIT_MIN} min: {list(pending)} — rulează din nou celulele 1, 2 și 4 mai târziu.')
        break
    if pending:
        time.sleep(60)

13:40 SWF_2021_005m 21780887710 Finished_ok


13:40 WVL_2021_005m 97973866919 Finished_ok


In [5]:
# Verificare: rasterele descărcate, CRS, rezoluție și acoperirea AOI
import rasterio
from rasterio.warp import transform_bounds
for p in sorted(SLF_DIR.rglob('*.tif')):
    with rasterio.open(p) as s:
        b = transform_bounds(s.crs, 'EPSG:4326', *s.bounds)
        acopera = b[0] <= W + 0.01 and b[1] <= S + 0.01 and b[2] >= E - 0.01 and b[3] >= N - 0.01
        print(p.relative_to(SLF_DIR), s.crs, s.res, (s.width, s.height), s.dtypes[0], 'nodata', s.nodata,
              '| bounds 4326', [round(v, 3) for v in b], '| acoperă AOI:', acopera)

SWF_2021_005m\Results\SWF_2021_005m_eu_03035_V01_R02\SWF_2021_005m_eu_03035_V01_R02\SWF_2021_005m_eu_03035_V01_R02.tif PROJCS["ETRS89-extended / LAEA Europe",GEOGCS["ETRS89",DATUM["European_Terrestrial_Reference_System_1989",SPHEROID["GRS 1980",6378137,298.257222101004,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6258"]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4258"]],PROJECTION["Lambert_Azimuthal_Equal_Area"],PARAMETER["latitude_of_center",52],PARAMETER["longitude_of_center",10],PARAMETER["false_easting",4321000],PARAMETER["false_northing",3210000],UNIT["metre",1],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","3035"]] (4.99999999943088, 5.000000000569163) (12238, 8690) uint8 nodata 255.0 | bounds 4326 [23.779, 43.606, 24.624, 44.094] | acoperă AOI: True
WVL_2021_005m\Results\WVL_2021_005m_03035_V01_R02\WVL_2021_005m_03035_V01_R02\WVL_2021_005m_03035_V01_R02.tif PROJCS["ETRS89-extended / LAEA Europe",GEOGCS["ETRS8